**Imports & Settings**

In [1]:
import pandas as pd
import re
import numpy as np 

pd.options.display.float_format = '{:.2f}'.format 

**Part 1 - Cleaning for 1st Dataset (airlines_review.csv)**

Previewing 1st Dataset

In [2]:
# Loading CSV
df1 = pd.read_csv("airlines_review.csv")

print(df1.head())


          Airlines         Name  Location Date Published  \
0  british_airways      C Hayne      2024     2024-08-15   
1  british_airways    C Porter       2024     2024-08-12   
2  british_airways      G Jones      2024     2024-08-12   
3  british_airways  Edward King      2024     2024-08-11   
4  british_airways       N Kwok      2024     2024-08-09   

                                        Text Content       Seat Type  \
0  Not Verified | Before my flight, I was forced ...    Solo Leisure   
1  ✅ Trip Verified |   British Airways at its bes...            A350   
2  ✅ Trip Verified | An excellent flight! Despite...            A320   
3  ✅ Trip Verified | I recently traveled with Bri...            A380   
4  ✅ Trip Verified |   My family and I were booke...  Family Leisure   

   Seat Comfort  Cabin Staff Service  Food & Beverages  \
0           NaN                  NaN               NaN   
1           NaN                  NaN               NaN   
2           NaN                 

1.1 - Remove Unecessary Columns (Data Cleaning)

In [3]:
df1.drop(columns=["Name", "Seat Type", "Location"], inplace=True, errors="ignore")

1.2 - Rename Columns (Data Standardization)

In [4]:
df1.rename(columns={
    "Airlines": "airline",
    "Name": "name",  
    "Date Published": "review_date",
    "Text Content": "review_text",
    "Seat Comfort": "seat_comfort",
    "Cabin Staff Service": "staff_service",
    "Food & Beverages": "food_beverage",
    "Inflight Entertainment": "inflight_entertainment",
    "Value For Money": "value_for_money",
    "Recommended": "recommended",
    "Ground Service": "ground_service",
    "Wifi & Connectivity": "wifi_connectivity"
}, inplace=True)

print(df1.columns)

Index(['airline', 'review_date', 'review_text', 'seat_comfort',
       'staff_service', 'food_beverage', 'inflight_entertainment',
       'value_for_money', 'recommended', 'ground_service',
       'wifi_connectivity'],
      dtype='object')


1.3 - Standardizing Airline Column (Data Standardization)

In [5]:
df1["airline"] = df1["airline"].str.lower().str.replace(" ", "_").str.replace("-", "_")

print(df1["airline"].unique())

['british_airways' 'air_france' 'emirates' 'qatar_airways'
 'qantas_airways' 'singapore_airlines']


1.4 - Clean "review_text" & Extract Verification Status (Data Transformation)

In [6]:
# Extract "Trip Verified" / "Not Verified" into a new column
df1["verification"] = df1["review_text"].str.extract(r"(Trip Verified|Not Verified)", expand=False)

#  Replace values with standardized format
df1["verification"] = df1["verification"].replace({
    "Trip Verified": "verified",
    "Not Verified": "not verified"
})

# Remove the verification phrase from the original review text
df1["review_text"] = df1["review_text"].str.replace(r"(Trip Verified|Not Verified)\s*\|\s*", "", regex=True)

# Convert to string 
df1["review_text"] = df1["review_text"].astype(str)

# Remove noisy characters
df1["review_text"] = df1["review_text"].str.encode('ascii', errors='ignore').str.decode('ascii')

# Remove extra spaces
df1["review_text"] = df1["review_text"].str.strip()
df1["review_text"] = df1["review_text"].str.replace(r"\s+", " ", regex=True)

# Preview cleaned results
print(df1[["verification", "review_text"]].head())

   verification                                        review_text
0  not verified  Before my flight, I was forced by the ground s...
1      verified  British Airways at its best. Outstanding servi...
2      verified  An excellent flight! Despite this being a 4.5 ...
3      verified  I recently traveled with British Airways and h...
4      verified  My family and I were booked to leave London fo...


1.5 - Convert Date Format & Separate Columns for "review_date" column (Data Transformation)

In [7]:
# Convert review_date to datetime format
df1["review_date"] = pd.to_datetime(df1["review_date"], errors="coerce") 

# Extract year, month, day as separate numeric columns for ML
df1["review_year"] = df1["review_date"].dt.year 
df1["review_month"] = df1["review_date"].dt.month
df1["review_day"] = df1["review_date"].dt.day

# Remove review_date since it's unused
df1.drop(columns=["review_date"], inplace=True, errors="ignore")

print(df1[["review_year", "review_month", "review_day"]].head())

   review_year  review_month  review_day
0         2024             8          15
1         2024             8          12
2         2024             8          12
3         2024             8          11
4         2024             8           9


1.6 - Cleaning Categorical Data (Data Normalization)

In [8]:
# Numerical and categorical columns to handle
columns_to_clean = [
    "seat_comfort",
    "staff_service",
    "food_beverage",
    "inflight_entertainment",
    "value_for_money",
    "ground_service",
    "wifi_connectivity",
    "recommended"          # boolean
]

# Replace blank/whitespace strings with NaN and round
for col in columns_to_clean:
    if col != "recommended":
        df1[col] = df1[col].replace(r'^\s*$', np.nan, regex=True)
        df1[col] = pd.to_numeric(df1[col], errors='coerce')
        df1[col] = df1[col].round(2)

# To forcefully show 2 digits in terminal
pd.options.display.float_format = '{:.2f}'.format

# Convert True/False to "yes"/"no" for consistency
df1["recommended"] = df1["recommended"].apply(lambda x: "yes" if x == True else "no")

print(df1[columns_to_clean].head())

   seat_comfort  staff_service  food_beverage  inflight_entertainment  \
0           NaN            NaN            NaN                    1.00   
1           NaN            NaN            NaN                     NaN   
2           NaN            NaN            NaN                     NaN   
3           NaN            NaN            NaN                     NaN   
4           NaN            NaN            NaN                    1.00   

   value_for_money  ground_service  wifi_connectivity recommended  
0             1.00             NaN                NaN          no  
1             5.00             NaN                NaN          no  
2             3.00             NaN                NaN          no  
3             3.00             NaN                NaN          no  
4             1.00             NaN                NaN          no  


1.7 - Exporting Cleaned 1st Dataset

In [9]:
df1.to_csv("cleaned_data1.csv", index=False, na_rep="NaN")

**Part 2 - Cleaning for 2nd Dataset (airlines_reviews.csv)**

Preview 2nd Dataset

In [10]:
# Load the dataset
df2 = pd.read_csv("airlines_reviews.csv")

print(df2.head())

                                  Title              Name Review Date  \
0                    Flight was amazing  Alison Soetantyo    3/1/2024   
1  seats on this aircraft are dreadful      Robert Watson   2/21/2024   
2          Food was plentiful and tasty             S Han   2/20/2024   
3          “how much food was available          D Laynes   2/19/2024   
4       “service was consistently good”         A Othman    2/19/2024   

              Airline Verified  \
0  Singapore Airlines     TRUE   
1  Singapore Airlines     TRUE   
2  Singapore Airlines     TRUE   
3  Singapore Airlines     TRUE   
4  Singapore Airlines     TRUE   

                                             Reviews Type of Traveller  \
0    Flight was amazing. The crew onboard this fl...      Solo Leisure   
1    Booking an emergency exit seat still meant h...      Solo Leisure   
2    Excellent performance on all fronts. I would...    Family Leisure   
3   Pretty comfortable flight considering I was f...      So

2.1 - Remove Unnecessary Columns (Data Cleaning)

In [11]:
df2.drop(columns=["Title", "Name", "Type of Traveller", "Route", "Class", "Month Flown"], inplace=True, errors="ignore")

2.2 - Rename Columns (Data Standardization)

In [12]:
df2.rename(columns={
    "Review Date": "review_date",
    "Airline": "airline",
    "Verified": "verification",
    "Reviews": "review_text",
    "Seat Comfort": "seat_comfort",
    "Staff Service": "staff_service",
    "Food & Beverages": "food_beverage",
    "Inflight Entertainment": "inflight_entertainment",
    "Value For Money": "value_for_money",
    "Overall Rating": "overall_rating",
    "Recommended": "recommended"
}, inplace=True)


print(df2.columns)

Index(['review_date', 'airline', 'verification', 'review_text', 'seat_comfort',
       'staff_service', 'food_beverage', 'inflight_entertainment',
       'value_for_money', 'overall_rating', 'recommended'],
      dtype='object')


2.3 - Standardizing Airlines Column (Data Standardization)

In [13]:
# To ensure that it is in string format by removing extra whitespaces
df2["airline"] = df2["airline"].str.lower().str.replace(" ", "_").str.replace("-", "_")

print(df2["airline"].head())

0    singapore_airlines
1    singapore_airlines
2    singapore_airlines
3    singapore_airlines
4    singapore_airlines
Name: airline, dtype: object


2.4 - Convert Date Format & Separate Columns for "review_date" column (Data Transformation)

In [14]:
# Convert review_date to datetime format
df2["review_date"] = pd.to_datetime(df2["review_date"], errors="coerce") 

# Extract year, month, day as separate columns
df2["review_year"] = df2["review_date"].dt.year 
df2["review_month"] = df2["review_date"].dt.month
df2["review_day"] = df2["review_date"].dt.day

# Remove review_date column since not needed
df2.drop(columns=["review_date", ""], inplace=True, errors="ignore") 

print(df2[["review_year", "review_month", "review_day"]].head())

   review_year  review_month  review_day
0         2024             3           1
1         2024             2          21
2         2024             2          20
3         2024             2          19
4         2024             2          19


2.5 - Standardize Verified Column

In [15]:
# To ensure verification text is lowercase
df2["verification"] = df2["verification"].astype(str).str.strip().str.lower()

# Set "true = verified" & "false = not verified"
df2["verification"] = df2["verification"].apply(lambda x: "verified" if x == "true" else "not verified")

print(df2[["verification"]].head())

  verification
0     verified
1     verified
2     verified
3     verified
4     verified


2.6 - Clean review_text (Text Cleaning)

In [16]:
# Convert to string 
df2["review_text"] = df2["review_text"].astype(str)

# Remove noisy characters
df2["review_text"] = df2["review_text"].str.encode('ascii', errors='ignore').str.decode('ascii')

# Remove unnecessary spaces
df2["review_text"] = df2["review_text"].str.strip()
df2["review_text"] = df2["review_text"].str.replace(r"\s+", " ", regex=True)

print(df2[["review_text"]].head())

                                         review_text
0  Flight was amazing. The crew onboard this flig...
1  Booking an emergency exit seat still meant hug...
2  Excellent performance on all fronts. I would d...
3  Pretty comfortable flight considering I was fl...
4  The service was consistently good from start t...


2.7 - Handling Categorical Data (Data Normalization & Transformation)

In [17]:
# Define the rating columns and their original ranges
rating_columns_df2 = {
    "seat_comfort": "1-5",
    "staff_service": "1-5",
    "food_beverage": "1-5",
    "inflight_entertainment": "0-5",  # already in correct scale
    "value_for_money": "1-10",
    "overall_rating": "1-10"
}

# Apply transformations
for col, scale in rating_columns_df2.items():
    if col in df2.columns:
        if scale == "1-5":
            # Convert 1–5 to 0–5 scale
            df2[col] = df2[col].apply(lambda x: round((x - 1) * (5 / 4), 2))
        elif scale == "1-10":
            # Convert 1–10 to 0–5 scale
            df2[col] = df2[col].apply(lambda x: round((x - 1) * (5 / 9), 2))
        # No change needed for "0-5" scale
        df2[col] = df2[col].round(2)  # Round to 2 decimal places

print(df2[["seat_comfort", "staff_service", "food_beverage", "inflight_entertainment", "value_for_money", "overall_rating"]].head())

   seat_comfort  staff_service  food_beverage  inflight_entertainment  \
0          3.75           3.75           3.75                       4   
1          5.00           2.50           3.75                       4   
2          0.00           5.00           1.25                       1   
3          5.00           5.00           5.00                       5   
4          5.00           5.00           5.00                       5   

   value_for_money  overall_rating  
0             1.67            4.44  
1             0.00            1.11  
2             2.22            5.00  
3             2.22            5.00  
4             2.22            5.00  


2.8 - Normalization for Recommended Columns

In [18]:
# To ensure lower case 
df2["recommended"] = df2["recommended"].str.strip().str.lower()

print(df2[["recommended"]].head())

  recommended
0         yes
1          no
2         yes
3         yes
4         yes


2.9 - Export Cleaned 2nd Dataset

In [19]:
df2.to_csv("cleaned_data2.csv", index=False, na_rep="NaN")

**Part 3 - Combining Datasets**

In [20]:
# Load both cleaned datasets
df1 = pd.read_csv("cleaned_data1.csv")
df2 = pd.read_csv("cleaned_data2.csv")

# Combine datasets
combined_df = pd.concat([df1, df2], ignore_index=True)

# reorder columns
column_order = [
    'review_year', 'review_month', 'review_day',
    'airline', 'review_text',
    'seat_comfort', 'staff_service', 'food_beverage',
    'inflight_entertainment', 'ground_service', 'wifi_connectivity',
    'value_for_money', 'recommended', 'verification',
    'overall_rating'
]
combined_df = combined_df[column_order]

**Part 4 - Clean Conbined "review_text"**

4.1 - Ensuring imports 

In [21]:
# Ensuring imports
import pandas as pd
import re
import string
from collections import Counter
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# Download stopwords (to ensure)
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')


# English stopword list
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()



[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\JulianYip\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\JulianYip\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\JulianYip\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


4.2 - Performing Basic Cleaning

In [22]:
import re
import string
import pandas as pd
from nltk.corpus import stopwords
import nltk

# Download stopwords if not already
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

review_col = "review_text"

def preprocess_text(text):
    if pd.isnull(text):
        text = ""
    
    # 1. Lowercase
    text = str(text).lower()
    
    # 2. Replace common separators with space
    text = re.sub(r'[\r\n\t]+', ' ', text)
    
    # 3. Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))
    
    # 4. Collapse multiple spaces
    text = re.sub(r'\s+', ' ', text).strip()
    
    # 5. Tokenize by splitting
    words = text.split()
    
    # 6. Remove stopwords
    words = [w for w in words if w not in stop_words]
    
    # 7. Rejoin words
    return " ".join(words)

# Apply to the dataframe
combined_df['cleaned_review_text'] = combined_df[review_col].apply(preprocess_text)

# Quick check
all_words = " ".join(combined_df['cleaned_review_text']).split()
print("Top 30 words after combined preprocessing:")
print(Counter(all_words).most_common(30))


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\JulianYip\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Top 30 words after combined preprocessing:
[('flight', 37681), ('service', 17509), ('food', 12643), ('time', 12053), ('good', 11820), ('seat', 11570), ('crew', 10882), ('class', 9977), ('seats', 9713), ('staff', 9622), ('cabin', 9328), ('business', 8774), ('one', 8747), ('would', 7891), ('airlines', 7156), ('economy', 7150), ('flights', 7078), ('airline', 7004), ('us', 6933), ('emirates', 6778), ('hours', 6672), ('first', 6564), ('get', 6480), ('via', 6267), ('airport', 6240), ('singapore', 6114), ('experience', 6012), ('even', 5617), ('airways', 5513), ('passengers', 5501)]


4.3 - Checking Individual Tokens After Basic Cleaning 

In [23]:
# --- Save all unique words in vertical list ---
from collections import Counter
import pandas as pd

all_words = []
for text in combined_df['cleaned_review_text']:
    all_words.extend(text.split())

unique_words = sorted(set(all_words))  # remove duplicates + sort alphabetically

# Write to txt file (each word on its own line)
with open("unique_words_list.txt", "w", encoding="utf-8") as f:
    for word in unique_words:
        f.write(word + "\n")

print(f"Saved {len(unique_words)} unique words to unique_words_list.txt")


Saved 34017 unique words to unique_words_list.txt


4.4 - Regex (Regular Expressions)

In [24]:
def replace_patterns(text):
    if not text:
        return ""
    
# 1. convert time/duration/date patterns to placeholder "time_duration"
    # Time
    text = re.sub(r'\b\d{1,2}[:\.\-]\d{2}(am|pm)?\b', ' time_duration ', text)   # 06:30, 06.30
    text = re.sub(r'\b\d{1,4}(am|pm)\b', ' time_duration ', text)                # 0630am, 0130pm, 9am
    text = re.sub(r'\b\d{1,2}h\d{1,2}m?\b', ' time_duration ', text)             # 07h25, 4h30
    text = re.sub(r'\b\d+(h|hr|hrs|hour|hours|min|mins|minute|minutes|month|months|yr|yrs|year|years)\b', ' time_duration ', text) # 3months, 2yr
    
    # Date
    text = re.sub(r'\b(?:\d{1,2}(?:st|nd|rd|th)?[-/\s]?)?(jan|feb|mar|apr|may|jun|jul|aug|sep|oct|nov|dec)[a-z]*[-]?\d{0,4}\b', ' date_value ', text) # 10th Oct, Mar 2024, 25 Jul
    text = re.sub(r'\b\d{1,2}[-/\.]\d{1,2}[-/\.]\d{2,4}\b', ' date_value ', text)  # 09/02/2020
    
# 2. Convert price/currency to placeholder "price_value"
    # Price before currency code 
    text = re.sub(r'\b\d{1,6}(usd|us|eur|gbp|chf|cad|aud|inr|idr|myr|php|hkd|sgd|try|krw|jpy|¥|€|\$|rs|rmb|cny)\w*\b', ' price_value ', text) #100usd, 200eur

    # Currency symbol before number 
    text = re.sub(r'\b\$\s?\d{1,6}\b', ' price_value ', text) #$100
    
# 3. Convert units (weight/length/volume/height) to placeholder "weight_value" 
    # Units
    text = re.sub(r'\b\d+(kg|kgs|g|gram|grams|lb|lbs|ft|feet|foot|inch|inches|cm|mm|l|litre|litres|ml|mls)\w*\b', ' weight_value ', text)

    # Specifically "5ft10", "6ft2"
    text = re.sub(r'\b\dft\d{1,2}\b', ' weight_value ', text)
    
# 4. Convert seat numbers (with 1-3 letters) to placeholder "seat_number"
    text = re.sub(r'\b\d{1,3}[a-zA-Z]{1,3}\b', ' seat_number ', text)
    
# 5. Convert ordinal numbers to placeholder "number_ordinal"
    text = re.sub(r'\b\d+(st|nd|rd|th)\b', ' number_ordinal ', text)
    
# 6. Convert aircraft models (many variants) to placeholder "aircraft_model"
    # Aircraft models in format like a320, a320-200, a321neo, b737-800, 737max, 777-300er, 787-9
    text = re.sub(r'\b(?:a|b)\d{2,4}(?:[- ]?(?:neo|ulr|xlr|lr|er|max|s)?)?\b', ' aircraft_model ', text)

    # Combined variants like a380b777 or 777a330 -> break into aircraft_model
    text = re.sub(r'\b(?:a|b)\d{2,4}(?:[a-z]{1,6}\d{0,4})+\b', ' aircraft_model ', text)
    
# 7. Convert flight/tail/registration codes to placeholder "flight_code"
    # common registration prefixes: a6, 9v, vh, ja, hs, oe, tc, g-, hb
    text = re.sub(r'\b(?:[a-z]{1,3}-?\d{0,4}|\d+[a-z]{2,})\b', lambda m: ' flight_code ' if re.search(r'\d', m.group(0)) and re.search(r'[a-zA-Z]', m.group(0)) else m.group(0), text)
    
    # explicit generic flight numbers with 2-letter airline code + digits (QR578)
    text = re.sub(r'\b[a-zA-Z]{2}\d{2,4}\b', ' flight_code ', text)
    
# 8. Remove any leftover long numeric-only tokens
    text = re.sub(r'\b\d{2,}\b', ' ', text)
    
# 9. Remove multiple spaces
    text = re.sub(r'\s+', ' ', text).strip()
    return text

combined_df['cleaned_review_text'] = combined_df['cleaned_review_text'].apply(replace_patterns)

# Check top words after basic cleaning
all_words = " ".join(combined_df['cleaned_review_text']).split()
print("Top 30 after basic cleaning:")
print(Counter(all_words).most_common(30))



Top 30 after basic cleaning:
[('flight', 37681), ('service', 17509), ('food', 12643), ('time', 12053), ('good', 11820), ('seat', 11570), ('crew', 10882), ('date_value', 10002), ('class', 9977), ('seats', 9713), ('staff', 9622), ('cabin', 9328), ('business', 8774), ('one', 8747), ('would', 7891), ('airlines', 7156), ('economy', 7150), ('flights', 7078), ('airline', 7004), ('us', 6933), ('emirates', 6778), ('hours', 6672), ('first', 6564), ('get', 6480), ('via', 6267), ('airport', 6240), ('singapore', 6114), ('experience', 6012), ('even', 5617), ('airways', 5513)]


4.5 - Convert multi-word entities to single tokens

In [25]:
def preserve_entities(text):
    if not isinstance(text, str):  # handle None/NaN
        text = ""
    text = text.lower()  # ensure lowercase for matching
    
    replacements = {
        "hong kong": "hong_kong",
        "new york": "new_york",
        "kuala lumpur": "kuala_lumpur",
        "london heathrow": "london_heathrow",
        "charles de gaulle": "charles_de_gaulle",
        "british airways": "british_airways",
        "singapore airlines": "singapore_airlines",
        "qatar airways": "qatar_airways",
        "emirates airlines": "emirates_airlines"
    }
    
    for phrase, token in replacements.items():
        text = text.replace(phrase, token)
    return text


# Check
all_words = " ".join(combined_df['cleaned_review_text']).split()
print("Top 30 after preserve_entities:")
print(Counter(all_words).most_common(30))


Top 30 after preserve_entities:
[('flight', 37681), ('service', 17509), ('food', 12643), ('time', 12053), ('good', 11820), ('seat', 11570), ('crew', 10882), ('date_value', 10002), ('class', 9977), ('seats', 9713), ('staff', 9622), ('cabin', 9328), ('business', 8774), ('one', 8747), ('would', 7891), ('airlines', 7156), ('economy', 7150), ('flights', 7078), ('airline', 7004), ('us', 6933), ('emirates', 6778), ('hours', 6672), ('first', 6564), ('get', 6480), ('via', 6267), ('airport', 6240), ('singapore', 6114), ('experience', 6012), ('even', 5617), ('airways', 5513)]


4.6 - Expanding shortforms

In [26]:
# Cell 5 - shortform expansion
shortform_dict = {
    'svc': 'service',
    'dep': 'departure',
    'arr': 'arrival',
    'bkk': 'bangkok',
    'sin': 'singapore',
    'kul': 'kuala_lumpur',
    'hkg': 'hong_kong',
    'lhr': 'london_heathrow',
    'cdg': 'charles_de_gaulle',
    'fco': 'rome',
    'zrh': 'zurich',
    'jfk': 'new_york',
    'lgw': 'london_gatwick',
    'us': 'united_states',
    'uk': 'united_kingdom',
    'ba': 'british_airways',
    'ek': 'emirates',
    'sq': 'singapore_airlines',
    'qr': 'qatar_airways',
    'cx': 'cathay_pacific'
}

def expand_shortforms(text):
    if not isinstance(text, str):  # handle None/NaN
        text = ""
    words = []
    for w in text.split():
        replacement = shortform_dict.get(w.lower(), w)  # lowercase for consistent matching
        words.append(replacement)
    return " ".join(words)

combined_df['cleaned_review_text'] = combined_df['cleaned_review_text'].apply(expand_shortforms)


# Check
all_words = " ".join(combined_df['cleaned_review_text']).split()
print("Top 30 after shortform expansion:")
print(Counter(all_words).most_common(30))


Top 30 after shortform expansion:
[('flight', 37681), ('service', 17509), ('food', 12643), ('time', 12053), ('good', 11820), ('seat', 11570), ('crew', 10882), ('date_value', 10002), ('class', 9977), ('seats', 9713), ('staff', 9622), ('cabin', 9328), ('business', 8774), ('one', 8747), ('would', 7891), ('emirates', 7183), ('airlines', 7156), ('economy', 7150), ('flights', 7078), ('airline', 7004), ('united_states', 6933), ('hours', 6672), ('first', 6564), ('get', 6480), ('singapore', 6419), ('via', 6267), ('airport', 6240), ('experience', 6012), ('even', 5617), ('airways', 5513)]


4.7 - Lemmatization & Remove Stop Words

In [27]:
def lemmatize_text_final(text):
    words = text.split()
    lemmas = [lemmatizer.lemmatize(w) for w in words]
    return " ".join(lemmas)

def remove_stopwords(text):
    tokens = text.split()
    filtered = [w for w in tokens if w not in stop_words]
    return " ".join(filtered)

combined_df['cleaned_review_text'] = combined_df['cleaned_review_text'].apply(remove_stopwords)

4.8 - Final Cleanup

In [28]:
def final_cleanup(text):
    tokens = []
    for t in text.split():
        # keep our placeholders and words with only letters/underscores
        if t in {"time_duration","price_value","weight_value","seat_number","aircraft_model","flight_code","date_value","number_ordinal","year_value"}:
            tokens.append(t)
        elif re.fullmatch(r'[a-z_]+', t):  # only lowercase letters and underscore
            tokens.append(t)
        # otherwise drop the token (it contained digits or weird chars)
    return " ".join(tokens)

combined_df['cleaned_review_text'] = combined_df['cleaned_review_text'].apply(final_cleanup)

# Top 50 to inspect
all_words = " ".join(combined_df['cleaned_review_text']).split()
word_counts = Counter(all_words)
print("Top 50 after final cleanup:")
print(word_counts.most_common(50))

# Export unique tokens to text file
unique_words = sorted(set(all_words))
with open("new_unique_words.txt", "w", encoding="utf-8") as f:
    for w in unique_words:
        f.write(w + "\n")




Top 50 after final cleanup:
[('flight', 37681), ('service', 17509), ('food', 12643), ('time', 12053), ('good', 11820), ('seat', 11570), ('crew', 10882), ('date_value', 10002), ('class', 9977), ('seats', 9713), ('staff', 9622), ('cabin', 9328), ('business', 8774), ('one', 8747), ('would', 7891), ('emirates', 7183), ('airlines', 7156), ('economy', 7150), ('flights', 7078), ('airline', 7004), ('united_states', 6933), ('hours', 6672), ('first', 6564), ('get', 6480), ('singapore', 6419), ('via', 6267), ('airport', 6240), ('experience', 6012), ('even', 5617), ('airways', 5513), ('passengers', 5501), ('aircraft_model', 5449), ('back', 5439), ('dubai', 5406), ('meal', 5387), ('doha', 5297), ('plane', 5233), ('boarding', 5208), ('could', 5201), ('great', 5147), ('british_airways', 5073), ('qatar', 4875), ('lounge', 4763), ('air', 4733), ('comfortable', 4636), ('new', 4599), ('return', 4507), ('fly', 4491), ('entertainment', 4323), ('aircraft', 4320)]


**Part 5 - Export Cleaned Overall Dataset**

In [29]:
combined_df.to_csv("combined_reviews.csv", index=False, na_rep="NaN")